# 02b — LLM Knowledge Extraction
Uses the local LLM to mine training pairs from real Examples, Book chapters, and Proposals.
All generated ARO code is validated with `aro check`. Valid pairs are saved to
`knowledge_pairs.jsonl` and used to warm-start fine-tune the model before notebook 03.

**Input:**  `../data/02_knowledge/knowledge.json`, `../../Examples/`, `../../Book/`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl`, `../data/adapters/warm_start/`

In [1]:
import json, re, sys, random, subprocess, tempfile
from pathlib import Path
from collections import Counter

def ensure_mlx_lm():
    try:
        from mlx_lm import load, generate as mlx_generate
        from mlx_lm.sample_utils import make_sampler
        return load, mlx_generate, make_sampler
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlx-lm'], check=True)
        from mlx_lm import load, generate as mlx_generate
        from mlx_lm.sample_utils import make_sampler
        return load, mlx_generate, make_sampler

load, mlx_generate, make_sampler = ensure_mlx_lm()

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

ARO_ROOT    = (SCRIPT_DIR / '../../').resolve()
DATA_IN     = SCRIPT_DIR / '../data/02_knowledge'
DATA_OUT    = SCRIPT_DIR / '../data/02_knowledge'
PAIRS_FILE  = DATA_OUT / 'knowledge_pairs.jsonl'
ADAPTER_DIR = SCRIPT_DIR / '../data/adapters'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit'

with open(DATA_IN / 'knowledge.json') as f:
    kb = json.load(f)

print(f'ARO root: {ARO_ROOT}')
print(f'Knowledge base: {len(kb["actions"])} actions, {len(kb["examples"])} examples, {len(kb["proposals"])} proposals')
print(f'Loading {MODEL_ID}...')
model, tokenizer = load(MODEL_ID)
print('Model ready.')

/Library/Python/3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/kris/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ARO root: /Users/kris/Projects/ARO-App
Knowledge base: 19 actions, 103 examples, 59 proposals
Loading mlx-community/Qwen2.5-Coder-7B-Instruct-4bit...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 92748.74it/s]


Model ready.


In [2]:
# Build system prompt from action metadata
action_lines = []
for a in kb['actions']:
    if a['verbs']:
        v = '/'.join(a['verbs'][:3])
        p = ', '.join(a['prepositions'][:4])
        action_lines.append(f'- {v}  (role: {a["role"]}, prepositions: {p})')

SYSTEM_PROMPT = f"""You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {{
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }}

KEY RULES:
- Articles (a/an/the) are optional
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>, <order-total>
- Forbidden variable name prefixes: is-, with-, empty-
- Exactly ONE Application-Start per application
- openapi.yaml required for HTTP server; operationId must match feature set name
- Long-running apps: Keepalive the <application> for the <events>.
- Return statuses: <OK: status>, <Created: status>, <NotFound: status>
- HTTP path params: Extract the <id> from the <pathParameters: id>.
- Request body:     Extract the <data> from the <request: body>.

AVAILABLE ACTIONS:
{chr(10).join(action_lines[:40])}

Always wrap ARO code in ```aro ... ``` fences."""


def chat(messages, max_tokens=800):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=0.3),
    )


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:300]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'


def aro_run_quick(code, timeout=8):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'run', tmp], capture_output=True, text=True, timeout=timeout)
            return r.returncode == 0, (r.stdout + r.stderr).strip()[:200]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return True, 'server_timeout'


# Resume support
_done_keys = set()
_pair_count = 0
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    s = json.loads(line)
                    _done_keys.add((s['source'], s['instruction'][:80]))
                    _pair_count += 1
                except Exception:
                    pass
    print(f'Resuming — {_pair_count} pairs already saved')


def save_pair(source, instruction, output, score=1.0, metadata=None):
    key = (source, instruction[:80])
    if key in _done_keys:
        return False
    record = {
        'instruction': instruction,
        'output': output,
        'source': source,
        'score': score,
        'metadata': metadata or {},
    }
    with open(PAIRS_FILE, 'a') as f:
        f.write(json.dumps(record) + '\n')
    _done_keys.add(key)
    return True


print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
print('Helpers ready.')

System prompt: 1939 chars
Helpers ready.


## Phase 1 — Real Examples → Training Pairs

For each of the 103 real, validated ARO examples:
- Ask the LLM to write a natural language instruction describing what the application does
- The code itself is already valid (taken verbatim from the repository)
- Result: (instruction, real_code) pairs with score=1.0

This is the highest-quality training data because the code is guaranteed correct.

In [ ]:
def load_app_dir(app_path):
    """Load a single ARO application directory into the same dict shape as kb['examples']."""
    p = Path(app_path)
    aro_files = {}
    for f in sorted(p.rglob('*.aro')):
        rel = str(f.relative_to(p))
        aro_files[rel] = f.read_text()

    def read_opt(name):
        fp = p / name
        return fp.read_text() if fp.exists() else None

    return {
        'name':      p.name,
        'dir':       str(p),
        'aro_files': aro_files,
        'openapi':   read_opt('openapi.yaml'),
        'readme':    read_opt('README.md'),
        'expected':  None,
    }


# ── Collect all examples ─────────────────────────────────────────────────────
# 1. Examples already in the knowledge base (from ./Examples/ and any previously
#    discovered paths in notebook 01/02)
all_examples = list(kb['examples'])

# 2. Directly scan ARO-Application for multi-file production apps
ARO_APP_DIR = (ARO_ROOT.parent / 'ARO-Application').resolve()
if ARO_APP_DIR.exists():
    known_names = {e['name'] for e in all_examples}
    for app_subdir in sorted(ARO_APP_DIR.iterdir()):
        if not app_subdir.is_dir():
            continue
        if app_subdir.name in known_names:
            continue            # already in kb['examples'] — skip duplicate
        if not list(app_subdir.rglob('*.aro')):
            continue            # no ARO files
        all_examples.append(load_app_dir(app_subdir))
        print(f'  Added ARO-Application/{app_subdir.name}', flush=True)
else:
    print(f'  ARO-Application not found at {ARO_APP_DIR}')

print(f'\nTotal examples for Phase 1: {len(all_examples)} '
      f'({len(kb["examples"])} from knowledge base + '
      f'{len(all_examples) - len(kb["examples"])} from ARO-Application)')


# ── Generate instruction for each example ────────────────────────────────────
print(f'\n--- Phase 1: Real Examples → Training Pairs ---')
count = 0

for ex in all_examples:
    if not ex['aro_files']:
        continue

    source_key = f'example:{ex["name"]}'

    # Build a compact code representation for the LLM context window
    parts = []
    if ex.get('openapi'):
        parts.append(f'## openapi.yaml\n```yaml\n{ex["openapi"][:600]}\n```')
    for fname, code in list(ex['aro_files'].items())[:5]:   # cap at 5 files
        parts.append(f'## {fname}\n```aro\n{code[:1200]}\n```')
    full_repr = '\n\n'.join(parts)[:4000]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is a complete ARO application called "{ex["name"]}":\n\n{full_repr}\n\n'
            f'Write ONE clear, specific natural language instruction (2-3 sentences) that '
            f'describes exactly what this application does — specific enough that a developer '
            f'following the instruction would produce this code.\n\n'
            f'Reply with ONLY the instruction text. No markdown, no labels.'
        )},
    ]
    instruction = chat(messages, max_tokens=200).strip()
    instruction = re.sub(r'^(#+\s*|Instruction:\s*|\*\*[^*]+\*\*:\s*)', '', instruction, flags=re.MULTILINE).strip()
    instruction = instruction.split('\n')[0].strip()

    if not instruction or len(instruction) < 20:
        continue

    # Reconstruct full output (all .aro files + openapi)
    output_parts = []
    if ex.get('openapi'):
        output_parts.append(f'## openapi.yaml\n```yaml\n{ex["openapi"]}\n```')
    for fname, code in ex['aro_files'].items():
        output_parts.append(f'## {fname}\n```aro\n{code}\n```')
    output = '\n\n'.join(output_parts)

    if save_pair(source_key, instruction, output, score=1.0,
                 metadata={'example': ex['name'], 'has_openapi': bool(ex.get('openapi')),
                            'source_dir': ex.get('dir', '')}):
        count += 1
        tag = ' [ARO-App]' if ex['name'] in {e.name for e in ARO_APP_DIR.iterdir() if ARO_APP_DIR.exists() and e.is_dir()} else ''
        print(f'  [{count}]{tag} {ex["name"]}: {instruction[:85]}', flush=True)

print(f'\nPhase 1 done: {count} new pairs from real examples')


## Phase 2 — Book: TheLanguageGuide → Training Pairs

For each of the 47 Language Guide chapters:
- Read the chapter markdown
- Ask the LLM to generate 3 instruction→code pairs that demonstrate concepts in that chapter
- Validate each generated code block with `aro check`
- Save only validated pairs

The Language Guide covers every aspect of ARO syntax, so this phase teaches the model
the full breadth of the language.

In [ ]:
print(f'\n--- Phase 2: TheLanguageGuide → Training Pairs ---')
count = 0

guide_dir = ARO_ROOT / 'Book' / 'TheLanguageGuide'
chapters = sorted(guide_dir.glob('*.md'))   # chapters + appendices

for chapter_path in chapters:
    chapter_name = chapter_path.stem
    content = chapter_path.read_text()

    # Only process chapters with ARO code blocks
    code_blocks = extract_aro_blocks(content)
    if not code_blocks:
        print(f'  {chapter_name}: no ARO blocks, skipping', flush=True)
        continue

    content_trimmed = content[:4500]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is "{chapter_name}" from the ARO Language Guide:\n\n{content_trimmed}\n\n'
            f'Generate 3 distinct instruction → ARO code training pairs based on this chapter.\n'
            f'Each pair must demonstrate a DIFFERENT concept from the chapter.\n'
            f'The ARO code must be complete and use only syntax shown in the chapter.\n\n'
            f'Format EXACTLY like this:\n\n'
            f'### Pair 1\n'
            f'**Instruction:** <natural language instruction>\n'
            f'**Code:**\n'
            f'```aro\n<complete valid ARO code>\n```\n\n'
            f'### Pair 2\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```\n\n'
            f'### Pair 3\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```'
        )},
    ]

    output_text = chat(messages, max_tokens=1400)

    # Parse pairs from output
    pair_blocks = re.split(r'###\s*Pair\s*\d+', output_text)
    chapter_count = 0

    for block in pair_blocks[1:]:
        # Extract instruction
        instr_match = re.search(r'\*\*Instruction:\*\*\s*(.+?)(?=\n\*\*|\n```|\Z)', block, re.DOTALL)
        if not instr_match:
            instr_match = re.search(r'Instruction:\s*(.+?)(?=\n(?:Code:|```|\*\*)|\Z)', block, re.DOTALL)
        if not instr_match:
            continue
        instruction = instr_match.group(1).strip().replace('\n', ' ')

        codes = extract_aro_blocks(block)
        if not codes or not instruction or len(instruction) < 15:
            continue

        code = codes[0]
        check_ok, check_err = aro_check(code)
        if check_ok is False:
            short_err = check_err.split('\n')[0][:70]
            print(f'    x {chapter_name}: {short_err}', flush=True)
            continue

        score = 1.0 if check_ok is True else 0.8  # 0.8 if aro not available
        if save_pair(f'book:{chapter_name}', instruction, f'```aro\n{code}\n```',
                     score=score, metadata={'chapter': chapter_name}):
            chapter_count += 1
            count += 1
            print(f'  [{count}] {chapter_name}: {instruction[:80]}', flush=True)

    if chapter_count == 0:
        print(f'  {chapter_name}: 0 valid pairs extracted', flush=True)

print(f'\nPhase 2 done: {count} new pairs from Language Guide')


--- Phase 2: TheLanguageGuide → Training Pairs ---
    x AppendixA-ActionReference: main.aro:
    x AppendixA-ActionReference: main.aro:
    x AppendixA-ActionReference: main.aro:
  AppendixA-ActionReference: 0 valid pairs extracted
    x AppendixB-Prepositions: main.aro:
    x AppendixB-Prepositions: main.aro:
    x AppendixB-Prepositions: main.aro:
  AppendixB-Prepositions: 0 valid pairs extracted
    x AppendixC-Statements: main.aro:
    x AppendixC-Statements: main.aro:
    x AppendixC-Statements: main.aro:
  AppendixC-Statements: 0 valid pairs extracted
    x Chapter01-WhyARO: main.aro:
    x Chapter01-WhyARO: main.aro:
    x Chapter01-WhyARO: main.aro:
  Chapter01-WhyARO: 0 valid pairs extracted
    x Chapter02-MentalModel: main.aro:
    x Chapter02-MentalModel: main.aro:
    x Chapter02-MentalModel: main.aro:
  Chapter02-MentalModel: 0 valid pairs extracted
  Chapter03-GettingStarted: no ARO blocks, skipping
    x Chapter04-StatementAnatomy: main.aro:
    x Chapter04-StatementA

## Phase 3 — Book: AROByExample → Training Pairs

AROByExample builds a complete web crawler in ARO across 14 chapters.
Each chapter shows real, working ARO code for a specific concern (fetching, parsing, storing, etc.).
The LLM generates 2 training pairs per chapter from the complete worked examples.

In [ ]:
print(f'\n--- Phase 3: AROByExample → Training Pairs ---')
count = 0

aro_by_example_dir = ARO_ROOT / 'Book' / 'AROByExample'
chapters = sorted(aro_by_example_dir.glob('Chapter*.md'))

for chapter_path in chapters:
    chapter_name = chapter_path.stem
    content = chapter_path.read_text()
    code_blocks = extract_aro_blocks(content)
    if not code_blocks:
        continue

    content_trimmed = content[:4000]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is "{chapter_name}" from "ARO By Example" (a hands-on ARO tutorial):\n\n'
            f'{content_trimmed}\n\n'
            f'Generate 2 instruction → ARO code pairs from the examples in this chapter.\n\n'
            f'### Pair 1\n'
            f'**Instruction:** <what this code does>\n'
            f'**Code:**\n'
            f'```aro\n<the ARO code>\n```\n\n'
            f'### Pair 2\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```'
        )},
    ]

    output_text = chat(messages, max_tokens=1000)
    pair_blocks = re.split(r'###\s*Pair\s*\d+', output_text)

    for block in pair_blocks[1:]:
        instr_match = re.search(r'\*\*Instruction:\*\*\s*(.+?)(?=\n\*\*|\n```|\Z)', block, re.DOTALL)
        if not instr_match:
            continue
        instruction = instr_match.group(1).strip().replace('\n', ' ')
        codes = extract_aro_blocks(block)
        if not codes or len(instruction) < 15:
            continue

        code = codes[0]
        check_ok, check_err = aro_check(code)
        if check_ok is False:
            continue

        if save_pair(f'aro_by_example:{chapter_name}', instruction,
                     f'```aro\n{code}\n```',
                     score=1.0 if check_ok is True else 0.8,
                     metadata={'chapter': chapter_name}):
            count += 1
            print(f'  [{count}] {chapter_name}: {instruction[:80]}', flush=True)

print(f'\nPhase 3 done: {count} new pairs from AROByExample')

## Phase 4 — Proposals → Contextualized Training Pairs

The language proposals contain hundreds of valid ARO code blocks that demonstrate specific
language features. The LLM provides a natural language instruction for each code block,
turning them into (instruction, validated_code) training pairs.

Only code that passes `aro check` is saved.

In [ ]:
print(f'\n--- Phase 4: Proposals → Contextualized Pairs ---')
count = 0

for prop in kb['proposals']:
    if not prop.get('aro_code_blocks'):
        continue

    prop_name = prop['name']

    for i, code in enumerate(prop['aro_code_blocks'][:8]):  # cap per proposal
        if len(code.strip()) < 40:
            continue

        # Check if already done (use code hash as key)
        code_key = f'proposal:{prop_name}:{i}'

        # Validate first — don't waste LLM time on invalid code
        check_ok, check_err = aro_check(code)
        if check_ok is False:
            continue

        # Find the closest section heading/body for context
        context = ''
        for seed in prop.get('qa_seeds', []):
            if seed.get('code_examples') and code[:50] in '\n'.join(seed['code_examples']):
                context = f'Section "{seed["heading"]}": {seed["body"][:400]}'
                break
        if not context and prop.get('qa_seeds'):
            context = prop['qa_seeds'][0]['body'][:300]

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': (
                f'This ARO code example is from specification {prop_name}.\n\n'
                f'Context: {context}\n\n'
                f'```aro\n{code}\n```\n\n'
                f'Write ONE natural language instruction (1-2 sentences) that would '
                f'prompt a developer to write exactly this code. '
                f'Reply with ONLY the instruction text.'
            )},
        ]

        instruction = chat(messages, max_tokens=120).strip()
        instruction = re.sub(r'^(#+\s*|Instruction:\s*|\*\*[^*]+\*\*:\s*)', '', instruction).strip()
        instruction = instruction.split('\n')[0].strip()

        if not instruction or len(instruction) < 15:
            continue

        if save_pair(code_key, instruction, f'```aro\n{code}\n```',
                     score=1.0 if check_ok is True else 0.7,
                     metadata={'proposal': prop_name, 'block_index': i}):
            count += 1
            if count % 20 == 0:
                print(f'  [{count}] {prop_name}[{i}]: {instruction[:70]}', flush=True)

print(f'\nPhase 4 done: {count} new pairs from proposals')

## Summary & Save

In [ ]:
all_pairs = []
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    all_pairs.append(json.loads(line))
                except Exception:
                    pass

sources = Counter(p['source'].split(':')[0] for p in all_pairs)
scores  = Counter(round(p.get('score', 1.0), 1) for p in all_pairs)

print(f'\nTotal knowledge pairs: {len(all_pairs)}')
print('\nBy source:')
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:30s}: {n}')
print('\nBy score:')
for score, n in sorted(scores.items(), key=lambda x: -x[0]):
    print(f'  {score}: {n}')

## Warm-Start Fine-Tune

Fine-tune Qwen2.5-Coder on all extracted pairs so it understands ARO syntax
before notebook 03 starts generating synthetic data.

Uses 16 LoRA layers (vs 8 in the RL loop) for a deeper initial fit.
The adapter is saved to `../data/adapters/warm_start/` and notebook 03
will automatically load it.

In [ ]:
import gc

WARM_ADAPTER = ADAPTER_DIR / 'warm_start'
SFT_DIR      = SCRIPT_DIR / '../data/warm_start_sft'
SFT_DIR.mkdir(parents=True, exist_ok=True)

# Shuffle and split
random.shuffle(all_pairs)
split = max(1, int(len(all_pairs) * 0.1))

def pair_to_chat(p):
    return {'messages': [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': p['instruction']},
        {'role': 'assistant', 'content': p['output']},
    ]}

(SFT_DIR / 'valid.jsonl').write_text(
    '\n'.join(json.dumps(pair_to_chat(p)) for p in all_pairs[:split]))
(SFT_DIR / 'train.jsonl').write_text(
    '\n'.join(json.dumps(pair_to_chat(p)) for p in all_pairs[split:]))

n_train = len(all_pairs) - split
iters   = max(100, min(500, n_train * 3))

print(f'SFT data: {n_train} train, {split} valid')
print(f'Warm-start: {iters} steps, 16 LoRA layers')

cmd = [
    sys.executable, '-m', 'mlx_lm.lora',
    '--model',         MODEL_ID,
    '--data',          str(SFT_DIR),
    '--train',
    '--num-layers',    '16',
    '--iters',         str(iters),
    '--batch-size',    '2',
    '--learning-rate', '2e-4',
    '--adapter-path',  str(WARM_ADAPTER),
    '--use-chat-template',
    '--mask-prompt',
    '--save-every',    str(max(50, iters // 5)),
    '--val-batches',   '5',
]

print('\nFreeing model memory before training subprocess...')
del model, tokenizer
gc.collect()

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, timeout=7200)

if result.returncode == 0:
    print(f'\nWarm-start adapter saved to: {WARM_ADAPTER}')
else:
    print(f'\nFine-tune exited with code {result.returncode}')

In [ ]:
# Update knowledge.json so notebook 03 finds the warm-start adapter automatically
kb['warm_start_adapter']    = str(WARM_ADAPTER)
kb['knowledge_pairs_file']  = str(PAIRS_FILE)
kb['knowledge_pairs_count'] = len(all_pairs)

with open(DATA_IN / 'knowledge.json', 'w') as f:
    json.dump(kb, f, indent=2)

print('Updated knowledge.json')
print()
print('Next steps:')
print(f'  1. Run notebook 03 — it will auto-load the warm-start adapter')
print(f'  2. The RL explore loop in notebook 03 will continue fine-tuning from this base')
print(f'  3. Adapter path: {WARM_ADAPTER}')
print(f'  4. Training pairs available at: {PAIRS_FILE}')